<div align="center">

![Course](https://img.shields.io/badge/Course-Advanced%20Data%20Science%20Lab-blue?style=for-the-badge&logo=python&logoColor=white)
![Code](https://img.shields.io/badge/Code-MCA33PE21-informational?style=for-the-badge)
![Assignment](https://img.shields.io/badge/Assignment-04-success?style=for-the-badge)

---

# Advanced Data Science Lab [MCA33PE21]
## Practical Assignment - 04
### Supervised Learning - Regression and Classification Algorithms

---
<br>

| Field | Details |
|---|---|
| **Academic Year** | **2026 – 2027** |
| **Class / Division** | **SYMCA (Semester-I)** |
| **PRN** | **125M1H026** |
| **Student Name** | **Shantanu Suryawanshi** |
| **Submission Date** | **16-09-2026** |
</div>


## Implementation and Evaluation of Machine Learning Algorithms & K-Fold Cross-Validation

---

### Objective
This notebook implements and evaluates four supervised classification algorithms using the datasets specified in the assignment:

1. **Logistic Regression** — Heart Disease Prediction
2. **Decision Tree** — Telco Customer Churn Prediction
3. **Random Forest** — Credit Card Fraud Detection
4. **Support Vector Machine (SVM)** — Iris Flower Classification

The notebook then applies **Stratified K-Fold Cross-Validation** to the same models and reports the average evaluation metrics across folds.

> **Note:** Place the downloaded CSV datasets in a `datasets/` folder beside this notebook. The file paths can be changed in the configuration cells if your filenames are different.


## Notebook Structure

| Section | Work |
|---|---|
| 0 | Imports and common configuration |
| 1 | Logistic Regression — Heart Disease |
| 2 | Decision Tree — Telco Customer Churn |
| 3 | Random Forest — Credit Card Fraud |
| 4 | SVM — Iris |
| 5 | Model Comparison |
| 6 | K-Fold Cross-Validation |
| 7 | Final Findings and Insights |

### Evaluation Metrics
For every classification model, the following metrics are used:

- **Accuracy:** Overall proportion of correct predictions.
- **Precision:** Proportion of predicted positive cases that are actually positive.
- **Recall:** Proportion of actual positive cases correctly identified.
- **F1-score:** Harmonic mean of precision and recall.
- **Confusion Matrix:** Shows the counts of true positives, true negatives, false positives, and false negatives.


## 0. Imports and Common Configuration

All required libraries are imported at the beginning. Imports are separated by purpose so that the notebook remains easy to read and maintain.

In [ ]:
# Data handling
import os
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Reproducibility
RANDOM_STATE = 42

print("All required libraries imported successfully.")

In [ ]:
# Dataset locations
DATASET_FOLDER = "datasets"

HEART_DATASET_PATH = os.path.join(DATASET_FOLDER, "heart_disease_uci.csv")
TELCO_DATASET_PATH = os.path.join(DATASET_FOLDER, "telco_customer_churn.csv")
FRAUD_DATASET_PATH = os.path.join(DATASET_FOLDER, "creditcard.csv")

print("Dataset folder:", DATASET_FOLDER)
print("Heart dataset:", HEART_DATASET_PATH)
print("Telco dataset:", TELCO_DATASET_PATH)
print("Fraud dataset:", FRAUD_DATASET_PATH)

### Common Evaluation Helper

The following small function keeps the metric calculation consistent across models. The individual model sections still show the complete training, prediction, confusion matrix, and classification report steps explicitly.

In [ ]:
def display_classification_results(model_name, y_test, y_pred):
    """Display standard classification metrics and the confusion matrix."""

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print(f"\n{model_name}")
    print("-" * len(model_name))
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

# 1. Implement and Evaluate Various Machine Learning Algorithms

The four classification algorithms are implemented independently using the datasets specified in the question.

Each subsection follows the same workflow:

**Load → Inspect → Clean → Encode → Split → Scale if required → Train → Predict → Evaluate → Visualize**

## 1(a). Logistic Regression — Heart Disease Prediction

### Problem Description
The objective is to predict whether a person has heart disease from health-related features such as age, sex, chest-pain type, blood pressure, cholesterol, maximum heart rate, and other clinical measurements.

**Algorithm:** Logistic Regression  
**Dataset:** Heart Disease  
**Target:** `HeartDisease`

The `HeartDisease` column contains values from 0 to 4. For this binary classification task, the target is converted to:

- `0` → No Heart Disease
- `1` → Heart Disease present

Therefore, values greater than 0 are mapped to `1`.

Logistic Regression is suitable for binary classification and benefits from feature scaling when numerical variables have different ranges.

### Step 1 — Load and Inspect the Dataset

In [ ]:
heart_data = pd.read_csv(HEART_DATASET_PATH)

print("First five rows:")
display(heart_data.head())

print("\nDataset shape:", heart_data.shape)
print("\nColumn names:")
print(heart_data.columns.tolist())

print("\nData types:")
print(heart_data.dtypes)

### Step 2 — Check Missing Values and Target Distribution

In [ ]:
print("\nOriginal target value counts:")
print(heart_data["num"].value_counts().sort_index())

# Convert the original 0–4 target into binary classification.
heart_data["heart_disease"] = (heart_data["num"] > 0).astype(int)

print("\nBinary target value counts:")
print(heart_data["heart_disease"].value_counts().sort_index())

plt.figure(figsize=(6, 4))
sns.countplot(data=heart_data, x="heart_disease")
plt.title("Heart Disease Target Distribution")
plt.xlabel("Heart Disease (0 = No, 1 = Yes)")
plt.ylabel("Number of Records")
plt.show()

### Step 3 — Separate Features and Target

Categorical variables are one-hot encoded. Numerical variables are standardized because Logistic Regression uses numerical coefficients and is sensitive to feature scale.

In [ ]:
heart_features = heart_data.drop(columns=["id", "num", "heart_disease"])
heart_target = heart_data["heart_disease"]

In [ ]:

heart_numeric_columns = heart_features.select_dtypes(include=np.number).columns.tolist()
heart_categorical_columns = heart_features.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:", heart_numeric_columns)
print("Categorical features:", heart_categorical_columns)

In [ ]:
heart_numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

heart_categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

heart_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", heart_numeric_pipeline, heart_numeric_columns),
        ("categorical", heart_categorical_pipeline, heart_categorical_columns)
    ]
)

print("Heart disease preprocessing pipeline created.")

### Step 4 — Train/Test Split

In [ ]:
X_heart_train, X_heart_test, y_heart_train, y_heart_test = train_test_split(
    heart_features,
    heart_target,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=heart_target
)

print("Training records:", X_heart_train.shape[0])
print("Testing records :", X_heart_test.shape[0])

### Step 5 — Create and Train Logistic Regression

In [ ]:
heart_logistic_model = Pipeline(
    steps=[
        ("preprocessing", heart_preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]
)

heart_logistic_model.fit(X_heart_train, y_heart_train)

print("Logistic Regression model trained successfully.")

### Step 6 — Make Predictions and Evaluate

In [ ]:
y_heart_pred = heart_logistic_model.predict(X_heart_test)

heart_logistic_results = display_classification_results(
    "Logistic Regression — Heart Disease",
    y_heart_test,
    y_heart_pred
)

### Step 7 — Visualize the Confusion Matrix

In [ ]:
heart_cm = confusion_matrix(y_heart_test, y_heart_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(heart_cm, annot=True, fmt="d", cmap="Blues")
plt.title("Logistic Regression — Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

## 1(b). Decision Tree — Telco Customer Churn Prediction

### Problem Description
The objective is to predict whether a customer will churn using customer information such as tenure, contract type, monthly charges, total charges, payment method, and service-related attributes.

**Algorithm:** Decision Tree Classifier  
**Dataset:** Telco Customer Churn  
**Target:** `Churn`

Decision Trees can work with non-linear relationships and do not require numerical feature scaling. Categorical features still need to be encoded.

### Step 1 — Load and Inspect the Dataset

In [ ]:
telco_data = pd.read_csv(TELCO_DATASET_PATH)

print("First five rows:")
display(telco_data.head())

print("\nDataset shape:", telco_data.shape)
print("\nColumn names:")
print(telco_data.columns.tolist())

### Step 2 — Clean Data and Handle Missing Values

In [ ]:
# TotalCharges may be stored as text because some records contain blank values.
telco_data["TotalCharges"] = pd.to_numeric(
    telco_data["TotalCharges"],
    errors="coerce"
)

print("Missing values before handling:")
print(telco_data.isnull().sum()[telco_data.isnull().sum() > 0])

telco_data["TotalCharges"] = telco_data["TotalCharges"].fillna(
    telco_data["TotalCharges"].median()
)

telco_data = telco_data.drop(columns=["customerID"])

print("\nMissing values after handling:")
print(telco_data.isnull().sum().sum())

### Step 3 — Encode the Target and Inspect the Data

In [ ]:
telco_data["Churn"] = telco_data["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Target value counts:")
print(telco_data["Churn"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=telco_data, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
plt.show()

### Step 4 — Separate Features, Encode Categories, and Split Data

In [ ]:
telco_features = telco_data.drop(columns=["Churn"])
telco_target = telco_data["Churn"]

telco_numeric_columns = telco_features.select_dtypes(include=np.number).columns.tolist()
telco_categorical_columns = telco_features.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:", telco_numeric_columns)
print("Categorical features:", telco_categorical_columns)

In [ ]:
telco_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", telco_numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), telco_categorical_columns)
    ]
)

X_telco_train, X_telco_test, y_telco_train, y_telco_test = train_test_split(
    telco_features,
    telco_target,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=telco_target
)

print("Training records:", X_telco_train.shape[0])
print("Testing records :", X_telco_test.shape[0])

### Step 5 — Create and Train Decision Tree

In [ ]:
telco_tree_model = Pipeline(
    steps=[
        ("preprocessing", telco_preprocessor),
        ("classifier", DecisionTreeClassifier(
            max_depth=5,
            random_state=RANDOM_STATE
        ))
    ]
)

telco_tree_model.fit(X_telco_train, y_telco_train)

print("Decision Tree model trained successfully.")

### Step 6 — Predict and Evaluate

In [ ]:
y_telco_pred = telco_tree_model.predict(X_telco_test)

telco_tree_results = display_classification_results(
    "Decision Tree — Telco Churn",
    y_telco_test,
    y_telco_pred
)

In [ ]:
telco_cm = confusion_matrix(y_telco_test, y_telco_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(telco_cm, annot=True, fmt="d", cmap="Blues")
plt.title("Decision Tree — Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

## 1(c). Random Forest — Credit Card Fraud Detection

### Problem Description
The objective is to classify credit-card transactions as fraudulent or non-fraudulent.

**Algorithm:** Random Forest Classifier  
**Dataset:** Credit Card Fraud Detection  
**Target:** `Class`

The fraud dataset is highly imbalanced in typical versions of this dataset. Therefore, accuracy alone should not be treated as the only useful metric. Precision, recall, F1-score, and the confusion matrix are also reported.

Random Forest does not require feature scaling, so numerical features are used directly.

### Step 1 — Load and Inspect the Dataset

In [ ]:
fraud_data = pd.read_csv(FRAUD_DATASET_PATH)

print("First five rows:")
display(fraud_data.head())

print("\nDataset shape:", fraud_data.shape)

print("\nColumn names:")
print(fraud_data.columns.tolist())

### Step 2 — Check Missing Values and Class Imbalance

In [ ]:
print("Total missing values:", fraud_data.isnull().sum().sum())

print("\nFraud class distribution:")
print(fraud_data["Class"].value_counts())

print("\nFraud class percentage:")
print(fraud_data["Class"].value_counts(normalize=True).mul(100).round(3))

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=fraud_data, x="Class")
plt.title("Credit Card Fraud Class Distribution")
plt.xlabel("Class (0 = Normal, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.show()

### Step 3 — Separate Features and Target

In [ ]:
fraud_features = fraud_data.drop(columns=["Class"])
fraud_target = fraud_data["Class"]

print("Number of input features:", fraud_features.shape[1])
print("Number of target values:", fraud_target.shape[0])

### Step 4 — Train/Test Split

In [ ]:
X_fraud_train, X_fraud_test, y_fraud_train, y_fraud_test = train_test_split(
    fraud_features,
    fraud_target,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=fraud_target
)

print("Training records:", X_fraud_train.shape[0])
print("Testing records :", X_fraud_test.shape[0])

### Step 5 — Create and Train Random Forest

In [ ]:
fraud_random_forest = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

fraud_random_forest.fit(X_fraud_train, y_fraud_train)

print("Random Forest model trained successfully.")

### Step 6 — Predict and Evaluate

In [ ]:
y_fraud_pred = fraud_random_forest.predict(X_fraud_test)

fraud_random_forest_results = display_classification_results(
    "Random Forest — Credit Card Fraud",
    y_fraud_test,
    y_fraud_pred
)

In [ ]:
fraud_cm = confusion_matrix(y_fraud_test, y_fraud_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(fraud_cm, annot=True, fmt="d", cmap="Blues")
plt.title("Random Forest — Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

## 1(d). Support Vector Machine — Iris Flower Classification

### Problem Description
The objective is to classify an iris flower into one of three species:

- Setosa
- Versicolor
- Virginica

The classification is based on sepal and petal measurements.

**Algorithm:** Support Vector Machine (SVM)  
**Dataset:** Iris  
**Target:** Flower species

The Iris dataset is available directly through scikit-learn, so no external CSV is required. SVM is sensitive to feature scale, therefore StandardScaler is used.

### Step 1 — Load and Inspect the Iris Dataset

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()

iris_features = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

iris_target = pd.Series(
    iris.target,
    name="Species"
)

print("First five rows:")
display(iris_features.head())

print("\nTarget classes:")
print(iris.target_names)

print("\nDataset shape:", iris_features.shape)

### Step 2 — Visualize the Iris Measurements

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=iris_features,
    x="sepal length (cm)",
    y="petal length (cm)",
    hue=iris_target.map(dict(enumerate(iris.target_names)))
)
plt.title("Iris Dataset — Sepal Length vs Petal Length")
plt.xlabel("Sepal Length (cm)")
plt.ylabel("Petal Length (cm)")
plt.show()

### Step 3 — Train/Test Split

In [ ]:
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    iris_features,
    iris_target,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=iris_target
)

print("Training records:", X_iris_train.shape[0])
print("Testing records :", X_iris_test.shape[0])

### Step 4 — Create and Train SVM

In [ ]:
iris_svm_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf"))
    ]
)

iris_svm_model.fit(X_iris_train, y_iris_train)

print("SVM model trained successfully.")

### Step 5 — Predict and Evaluate

In [ ]:
y_iris_pred = iris_svm_model.predict(X_iris_test)

iris_svm_results = display_classification_results(
    "SVM — Iris Classification",
    y_iris_test,
    y_iris_pred
)

In [ ]:
iris_cm = confusion_matrix(y_iris_test, y_iris_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(iris_cm, annot=True, fmt="d", cmap="Blues")
plt.title("SVM — Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

# 2. Compare the Performance of Different Models

The four models use different datasets and targets, so their raw scores should **not** be interpreted as a direct scientific ranking of the algorithms. The comparison table is included to satisfy the assignment requirement and to show the measured results for each experiment.

In [ ]:
model_results = pd.DataFrame([
    heart_logistic_results,
    telco_tree_results,
    fraud_random_forest_results,
    iris_svm_results
])

print("Model Performance Comparison:")
display(model_results.round(4))

In [ ]:
metric_columns = ["Accuracy", "Precision", "Recall", "F1-score"]

model_results.set_index("Model")[metric_columns].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Classification Model Performance")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

## Findings and Insights — Part 1

1. **Logistic Regression** provides a linear decision boundary and is useful for binary heart-disease prediction after encoding and scaling.
2. **Decision Tree** can model non-linear relationships and does not require numerical scaling.
3. **Random Forest** combines multiple decision trees and is useful for complex classification problems. For fraud detection, the class imbalance makes precision, recall, F1-score, and the confusion matrix particularly important.
4. **SVM** works effectively with standardized numerical features and can classify multiple Iris species.
5. Because each algorithm is evaluated on a different dataset, the scores describe the performance of each **specific experiment**, not a universal ranking of the algorithms.

# 3. Implement K-Fold Cross-Validation Techniques

## Objective

Cross-validation evaluates a model across multiple train/validation splits instead of relying on only one train/test split.

For classification problems, **StratifiedKFold** is used so that the class distribution is approximately preserved in each fold.

The following configuration uses **5 folds**:

- Fold 1 → validation on fold 1, training on the remaining folds
- Fold 2 → validation on fold 2, training on the remaining folds
- ...
- Fold 5 → validation on fold 5, training on the remaining folds

Accuracy, precision, recall, and F1-score are calculated for every fold, followed by the mean across all five folds.

## 3(a). Logistic Regression with Stratified K-Fold Cross-Validation

The preprocessing is kept inside the Pipeline. This is important because scaling and one-hot encoding must be learned from the training portion of each fold rather than from the complete dataset.

In [ ]:
heart_cv_model = Pipeline(
    steps=[
        ("preprocessing", heart_preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]
)

heart_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("5-fold StratifiedKFold created for Logistic Regression.")

In [ ]:
heart_cv_scores = cross_validate(
    heart_cv_model,
    heart_features,
    heart_target,
    cv=heart_cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

heart_cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": heart_cv_scores["test_accuracy"],
    "Precision": heart_cv_scores["test_precision"],
    "Recall": heart_cv_scores["test_recall"],
    "F1-score": heart_cv_scores["test_f1"]
})

display(heart_cv_results.round(4))

In [ ]:
print("Average Cross-Validation Metrics:")
print(f"Mean Accuracy : {heart_cv_results['Accuracy'].mean():.4f}")
print(f"Mean Precision: {heart_cv_results['Precision'].mean():.4f}")
print(f"Mean Recall   : {heart_cv_results['Recall'].mean():.4f}")
print(f"Mean F1-score : {heart_cv_results['F1-score'].mean():.4f}")

## 3(b). Decision Tree with Stratified K-Fold Cross-Validation

In [ ]:
telco_cv_model = Pipeline(
    steps=[
        ("preprocessing", telco_preprocessor),
        ("classifier", DecisionTreeClassifier(
            max_depth=5,
            random_state=RANDOM_STATE
        ))
    ]
)

telco_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("5-fold StratifiedKFold created for Decision Tree.")

In [ ]:
telco_cv_scores = cross_validate(
    telco_cv_model,
    telco_features,
    telco_target,
    cv=telco_cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

telco_cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": telco_cv_scores["test_accuracy"],
    "Precision": telco_cv_scores["test_precision"],
    "Recall": telco_cv_scores["test_recall"],
    "F1-score": telco_cv_scores["test_f1"]
})

display(telco_cv_results.round(4))

In [ ]:
print("Average Cross-Validation Metrics:")
print(f"Mean Accuracy : {telco_cv_results['Accuracy'].mean():.4f}")
print(f"Mean Precision: {telco_cv_results['Precision'].mean():.4f}")
print(f"Mean Recall   : {telco_cv_results['Recall'].mean():.4f}")
print(f"Mean F1-score : {telco_cv_results['F1-score'].mean():.4f}")

In [ ]:
print("Fraud dataset shape:", fraud_features.shape)
print("Number of features:", fraud_features.shape[1])

## 3(c). Random Forest with Stratified K-Fold Cross-Validation

The credit-card dataset is strongly imbalanced. The Random Forest therefore uses `class_weight="balanced"` so that the minority fraud class receives greater importance during training.

In [ ]:
fraud_cv_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

fraud_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("5-fold StratifiedKFold created for Random Forest.")

In [ ]:
fraud_cv_scores = cross_validate(
    fraud_cv_model,
    fraud_features,
    fraud_target,
    cv=fraud_cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=1
)

fraud_cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": fraud_cv_scores["test_accuracy"],
    "Precision": fraud_cv_scores["test_precision"],
    "Recall": fraud_cv_scores["test_recall"],
    "F1-score": fraud_cv_scores["test_f1"]
})

display(fraud_cv_results.round(4))

In [ ]:
print("Average Cross-Validation Metrics:")
print(f"Mean Accuracy : {fraud_cv_results['Accuracy'].mean():.4f}")
print(f"Mean Precision: {fraud_cv_results['Precision'].mean():.4f}")
print(f"Mean Recall   : {fraud_cv_results['Recall'].mean():.4f}")
print(f"Mean F1-score : {fraud_cv_results['F1-score'].mean():.4f}")

## 3(d). SVM with Stratified K-Fold Cross-Validation

In [ ]:
iris_cv_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf"))
    ]
)

iris_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("5-fold StratifiedKFold created for SVM.")

In [ ]:
iris_cv_scores = cross_validate(
    iris_cv_model,
    iris_features,
    iris_target,
    cv=iris_cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

iris_cv_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": iris_cv_scores["test_accuracy"],
    "Precision": iris_cv_scores["test_precision"],
    "Recall": iris_cv_scores["test_recall"],
    "F1-score": iris_cv_scores["test_f1"]
})

display(iris_cv_results.round(4))

In [ ]:
print("Average Cross-Validation Metrics:")
print(f"Mean Accuracy : {iris_cv_results['Accuracy'].mean():.4f}")
print(f"Mean Precision: {iris_cv_results['Precision'].mean():.4f}")
print(f"Mean Recall   : {iris_cv_results['Recall'].mean():.4f}")
print(f"Mean F1-score : {iris_cv_results['F1-score'].mean():.4f}")

# 4. Cross-Validation Summary

The following table reports the mean performance across the five folds for each experiment.

In [ ]:
cross_validation_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression — Heart Disease",
        "Mean Accuracy": heart_cv_results["Accuracy"].mean(),
        "Mean Precision": heart_cv_results["Precision"].mean(),
        "Mean Recall": heart_cv_results["Recall"].mean(),
        "Mean F1-score": heart_cv_results["F1-score"].mean()
    },
    {
        "Model": "Decision Tree — Telco Churn",
        "Mean Accuracy": telco_cv_results["Accuracy"].mean(),
        "Mean Precision": telco_cv_results["Precision"].mean(),
        "Mean Recall": telco_cv_results["Recall"].mean(),
        "Mean F1-score": telco_cv_results["F1-score"].mean()
    },
    {
        "Model": "Random Forest — Credit Card Fraud",
        "Mean Accuracy": fraud_cv_results["Accuracy"].mean(),
        "Mean Precision": fraud_cv_results["Precision"].mean(),
        "Mean Recall": fraud_cv_results["Recall"].mean(),
        "Mean F1-score": fraud_cv_results["F1-score"].mean()
    },
    {
        "Model": "SVM — Iris",
        "Mean Accuracy": iris_cv_results["Accuracy"].mean(),
        "Mean Precision": iris_cv_results["Precision"].mean(),
        "Mean Recall": iris_cv_results["Recall"].mean(),
        "Mean F1-score": iris_cv_results["F1-score"].mean()
    }
])

display(cross_validation_summary.round(4))

In [ ]:
cross_validation_summary.set_index("Model").plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("5-Fold Cross-Validation Performance")
plt.xlabel("Model")
plt.ylabel("Mean Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

# 5. Final Findings and Insights

### Part 1 — Train/Test Evaluation
- The four classification algorithms were implemented using scikit-learn.
- Missing values were handled where required.
- Categorical features were converted using one-hot encoding.
- Numerical scaling was applied to Logistic Regression and SVM because these models are sensitive to feature magnitude.
- Decision Tree and Random Forest do not require feature scaling.
- Accuracy, precision, recall, F1-score, confusion matrix, and classification reports were used for evaluation.

### Part 2 — K-Fold Cross-Validation
- **StratifiedKFold with 5 folds** was used for all four classification experiments.
- Each fold was used once as a validation set while the other folds were used for training.
- Mean accuracy, precision, recall, and F1-score were calculated across all folds.
- Keeping preprocessing inside pipelines prevents information from the validation folds from leaking into preprocessing during cross-validation.

### Important Interpretation
The datasets represent different prediction problems with different feature spaces, class distributions, and levels of difficulty. Therefore, the numerical scores should be interpreted within their respective experiments rather than as a universal ranking of Logistic Regression, Decision Tree, Random Forest, and SVM.

### Conclusion
The notebook demonstrates a complete machine-learning classification workflow, from data preprocessing and model training to test-set evaluation and cross-validation. Cross-validation provides a more reliable estimate of how consistently each model performs across different data partitions.

---

<div align="center">

### Declaration

_I hereby declare that the work submitted in this practical assignment is my own and has not been copied from any other source._

| | |
|---|---|
| **Student Name** |Shantanu Suryawanshi |
| **PRN** | 125M1H026|
| **Date** | 16-09-2026|

---

**Prof. Prakash Ukhalkar**  
Course Teacher - Advanced Data Science Lab [MCA33PE21]

![](https://img.shields.io/badge/MCA33PE21-Advanced%20Data%20Science%20Lab-blue?style=flat-square)
![](https://img.shields.io/badge/Practical%20Assignment-01-green?style=flat-square)

</div>